In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
CLEANED_DIR = Path("../data/cleaned")
FORECAST_DIR = Path("../data/forecast")

DASHBOARD_DIR = Path("../data/dashboard")

DASHBOARD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Dashboard directory ready.")

Dashboard directory ready.


In [3]:
sales = pd.read_csv(
    CLEANED_DIR / "retail_sales.csv"
)

master = pd.read_csv(
    CLEANED_DIR / "retail_master_cleaned.csv"
)

rfm = pd.read_csv(
    CLEANED_DIR / "customer_rfm_segments.csv"
)

churn = pd.read_csv(
    CLEANED_DIR / "customer_churn_analysis.csv"
)

forecast = pd.read_csv(
    FORECAST_DIR / "30_day_demand_forecast.csv"
)

inventory = pd.read_csv(
    FORECAST_DIR / "inventory_planning.csv"
)

In [4]:
sales["InvoiceDate"] = pd.to_datetime(
    sales["InvoiceDate"],
    errors="coerce"
)

master["InvoiceDate"] = pd.to_datetime(
    master["InvoiceDate"],
    errors="coerce"
)

forecast["ds"] = pd.to_datetime(
    forecast["ds"],
    errors="coerce"
)

## Executive KPI dataset

In [6]:
total_revenue = sales["Revenue"].sum()

total_orders = sales["Invoice"].nunique()

total_customers = sales["Customer_ID"].nunique()

total_products = sales["StockCode"].nunique()

total_quantity = sales["Quantity"].sum()

average_order_value = (
    total_revenue / total_orders
)

In [7]:
executive_kpis = pd.DataFrame({
    "Metric": [
        "Total Revenue",
        "Total Orders",
        "Total Customers",
        "Total Products",
        "Total Quantity",
        "Average Order Value"
    ],
    
    "Value": [
        total_revenue,
        total_orders,
        total_customers,
        total_products,
        total_quantity,
        average_order_value
    ]
})

executive_kpis

,Metric,Value
0,Total Revenue,1.737480e+07
1,Total Orders,3.696900e+04
2,Total Customers,5.878000e+03
3,Total Products,4.631000e+03
4,Total Quantity,1.051395e+07
5,Average Order Value,4.699831e+02


## Monthly sales dataset

In [9]:
monthly_sales = (
    sales
    .set_index("InvoiceDate")
    .resample("ME")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Invoice", "nunique"),
        Quantity=("Quantity", "sum")
    )
    .reset_index()
)

monthly_sales

,InvoiceDate,Revenue,Orders,Quantity
0,2009-12-31,683504.010,1512,398660
1,2010-01-31,555802.672,1011,370082
2,2010-02-28,504558.956,1104,371861
3,2010-03-31,696978.471,1524,502100
4,2010-04-30,591982.002,1329,350587
5,2010-05-31,597833.380,1377,384960
6,2010-06-30,636371.130,1497,389872
7,2010-07-31,589736.170,1381,324632
8,2010-08-31,602224.600,1293,452542
9,2010-09-30,829013.951,1689,567153


## Country performance

In [12]:
country_performance = (
    sales
    .groupby("Country")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Invoice", "nunique"),
        Quantity=("Quantity", "sum")
    )
    .reset_index()
    .sort_values(
        "Revenue",
        ascending=False
    )
)

country_performance.head(20)

,Country,Revenue,Orders,Quantity
38,United Kingdom,1.438923e+07,33541,8532411
10,EIRE,6.165705e+05,567,318021
24,Netherlands,5.540381e+05,228,383879
14,Germany,4.250197e+05,789,225154
13,France,3.487690e+05,614,270288
0,Australia,1.692835e+05,95,103759
32,Spain,1.083325e+05,154,50307
34,Switzerland,1.000619e+05,90,52227
33,Sweden,9.151582e+04,104,88495
9,Denmark,6.858069e+04,43,237471


## Product performance

In [14]:
product_performance = (
    sales
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .agg(
        Revenue=("Revenue", "sum"),
        Quantity=("Quantity", "sum"),
        Orders=("Invoice", "nunique")
    )
    .reset_index()
    .sort_values(
        "Revenue",
        ascending=False
    )
)

product_performance.head(20)

,StockCode,Description,Revenue,Quantity,Orders
1862,22423,REGENCY CAKESTAND 3 TIER,277656.25,24124,3317
4751,85123A,WHITE HANGING HEART T-LIGHT HOLDER,247048.01,91757,4888
3283,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1
5309,M,Manual,151777.67,9384,620
4725,85099B,JUMBO BAG RED RETROSPOT,134307.44,74224,2612
5311,POST,POSTAGE,124648.04,5235,1803
4456,84879,ASSORTED COLOUR BIRD ORNAMENT,124351.86,78234,2652
3627,47566,PARTY BUNTING,103283.38,23460,2077
2781,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.73,77916,195
1449,22086,PAPER CHAIN KIT 50'S CHRISTMAS,76598.18,28380,1691


## Customer segment dataset

In [16]:
customer_segments = (
    rfm
    .groupby("Segment")
    .agg(
        Customers=("Customer_ID", "count"),
        Revenue=("Monetary", "sum"),
        Avg_Recency=("Recency", "mean"),
        Avg_Frequency=("Frequency", "mean"),
        Avg_Monetary=("Monetary", "mean")
    )
    .reset_index()
)

customer_segments

,Segment,Customers,Revenue,Avg_Recency,Avg_Frequency,Avg_Monetary
0,At Risk,824,1.589384e+06,369.100728,4.940534,1928.864456
1,Champions,1297,1.185959e+07,19.961449,17.105628,9143.863913
2,High Value Customers,255,7.571579e+05,60.364706,3.050980,2969.246518
3,Loyal Customers,701,1.856667e+06,84.368046,7.736091,2648.597753
4,New / Potential,417,1.760136e+05,28.503597,1.436451,422.095060
5,Other,2384,1.135990e+06,321.719379,1.640940,476.505845


## Churn summary

In [18]:
churn_summary = (
    churn
    .groupby("RiskLevel", observed=False)
    .agg(
        Customers=("Customer_ID", "count"),
        Revenue=("Monetary", "sum"),
        Avg_Churn_Probability=(
            "ChurnProbability",
            "mean"
        )
    )
    .reset_index()
)

churn_summary

,RiskLevel,Customers,Revenue,Avg_Churn_Probability
0,High Risk,2904,2.898140e+06,0.853171
1,Low Risk,2494,1.364723e+07,0.116657
2,Medium Risk,480,8.294347e+05,0.418142


## Churn by segment

In [20]:
churn_segment = (
    churn
    .groupby("Segment")
    .agg(
        Customers=("Customer_ID", "count"),
        Churned=("Churn", "sum"),
        Revenue=("Monetary", "sum")
    )
    .reset_index()
)

churn_segment["ChurnRate"] = (
    churn_segment["Churned"]
    / churn_segment["Customers"]
    * 100
)

churn_segment

,Segment,Customers,Churned,Revenue,ChurnRate
0,At Risk,824,824,1.589384e+06,100.000000
1,Champions,1297,0,1.185959e+07,0.000000
2,High Value Customers,255,56,7.571579e+05,21.960784
3,Loyal Customers,701,283,1.856667e+06,40.370899
4,New / Potential,417,0,1.760136e+05,0.000000
5,Other,2384,1826,1.135990e+06,76.593960


In [22]:
high_value_at_risk = (
    churn[
        churn["RiskLevel"] == "High Risk"
    ]
    .sort_values(
        "Monetary",
        ascending=False
    )
    .head(100)
)

In [24]:
high_value_at_risk = high_value_at_risk[
    [
        "Customer_ID",
        "Recency",
        "Frequency",
        "Monetary",
        "Segment",
        "ChurnProbability",
        "RiskLevel"
    ]
]

high_value_at_risk.head(20)

,Customer_ID,Recency,Frequency,Monetary,Segment,ChurnProbability,RiskLevel
4364,16754,372,29,65500.070,At Risk,0.740000,High Risk
3371,15749,235,3,44534.300,At Risk,0.806667,High Risk
2722,15098,182,3,39916.500,High Value Customers,0.906667,High Risk
1438,13802,139,19,26259.110,Loyal Customers,0.723333,High Risk
135,12482,576,29,23691.400,At Risk,0.786667,High Risk
1698,14063,438,9,22710.200,At Risk,0.770000,High Risk
678,13027,114,19,17239.200,Loyal Customers,0.756667,High Risk
3429,15808,306,21,17180.250,At Risk,0.650000,High Risk
4166,16553,163,33,16584.010,Loyal Customers,0.750000,High Risk
1203,13564,144,37,16498.970,Loyal Customers,0.743333,High Risk


## Forecast dashboard data

In [25]:
forecast_dashboard = forecast[
    [
        "ds",
        "ForecastDemand",
        "yhat_lower",
        "yhat_upper"
    ]
].copy()

forecast_dashboard.head()

,ds,ForecastDemand,yhat_lower,yhat_upper
0,2011-11-10,478.200560,-41.428225,979.993823
1,2011-11-11,336.175734,-130.869307,814.972084
2,2011-11-12,172.483840,-299.724836,653.852615
3,2011-11-13,230.113143,-273.743384,685.714171
4,2011-11-14,292.675554,-161.143692,766.103792


In [26]:
forecast_dashboard = forecast_dashboard.rename(
    columns={
        "ds": "Date",
        "ForecastDemand": "Forecast_Demand",
        "yhat_lower": "Lower_Bound",
        "yhat_upper": "Upper_Bound"
    }
)

## Inventory dashboard data

In [27]:
inventory_dashboard = inventory.copy()

inventory_dashboard

,Product,Average_Daily_Demand,Lead_Time_Days,Safety_Stock,Reorder_Point,Forecast_30_Day_Demand,Assumed_Current_Stock,Projected_30_Day_Stock,Recommendation
0,Selected Forecast Product,221.6859,7,1672.155268,3223.95657,6650.577009,1000,-5650.577009,Reorder immediately


## Save everything

In [28]:
executive_kpis.to_csv(
    DASHBOARD_DIR / "executive_kpis.csv",
    index=False
)

monthly_sales.to_csv(
    DASHBOARD_DIR / "monthly_sales.csv",
    index=False
)

country_performance.to_csv(
    DASHBOARD_DIR / "country_performance.csv",
    index=False
)

product_performance.to_csv(
    DASHBOARD_DIR / "product_performance.csv",
    index=False
)

customer_segments.to_csv(
    DASHBOARD_DIR / "customer_segments.csv",
    index=False
)

churn_summary.to_csv(
    DASHBOARD_DIR / "churn_summary.csv",
    index=False
)

churn_segment.to_csv(
    DASHBOARD_DIR / "churn_segment.csv",
    index=False
)

high_value_at_risk.to_csv(
    DASHBOARD_DIR / "high_value_at_risk.csv",
    index=False
)

forecast_dashboard.to_csv(
    DASHBOARD_DIR / "forecast_dashboard.csv",
    index=False
)

inventory_dashboard.to_csv(
    DASHBOARD_DIR / "inventory_dashboard.csv",
    index=False
)

print("All dashboard datasets saved successfully.")

All dashboard datasets saved successfully.


In [29]:
from pathlib import Path

print("Current notebook working directory:")
print(Path.cwd())

print("\nCSV files found:")
for file in Path.cwd().rglob("*.csv"):
    print(file)

Current notebook working directory:
C:\Users\alava\Downloads\Online_Retail_Project

CSV files found:


In [30]:
from pathlib import Path

BASE_DIR = Path.cwd()

RAW_DIR = BASE_DIR / "data" / "raw"
CLEANED_DIR = BASE_DIR / "data" / "cleaned"
FORECAST_DIR = BASE_DIR / "data" / "forecast"
DASHBOARD_DIR = BASE_DIR / "data" / "dashboard"

for folder in [
    RAW_DIR,
    CLEANED_DIR,
    FORECAST_DIR,
    DASHBOARD_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("Project directory:")
print(BASE_DIR)

print("\nFolders created:")
print(CLEANED_DIR)
print(FORECAST_DIR)
print(DASHBOARD_DIR)

Project directory:
C:\Users\alava\Downloads\Online_Retail_Project

Folders created:
C:\Users\alava\Downloads\Online_Retail_Project\data\cleaned
C:\Users\alava\Downloads\Online_Retail_Project\data\forecast
C:\Users\alava\Downloads\Online_Retail_Project\data\dashboard


In [31]:
print("CSV files inside project:")

for file in BASE_DIR.rglob("*.csv"):
    print(file)

CSV files inside project:


In [32]:
from pathlib import Path

print("Searching project and parent folders...\n")

search_locations = [
    Path.cwd(),
    Path.cwd().parent
]

for location in search_locations:
    print(f"\nSearching: {location}")
    
    for file in location.rglob("*.csv"):
        print(file)

Searching project and parent folders...


Searching: C:\Users\alava\Downloads\Online_Retail_Project

Searching: C:\Users\alava\Downloads
C:\Users\alava\Downloads\housing.csv
C:\Users\alava\Downloads\Long-Term Care Facility CharacteristicsForm_671_Q2_2026.csv
C:\Users\alava\Downloads\practicedataset.csv
C:\Users\alava\Downloads\sales.csv
C:\Users\alava\Downloads\salesdata.csv
C:\Users\alava\Downloads\SuperStore_Sales_Dataset (1).csv
C:\Users\alava\Downloads\Customer Transactions\unclean_customer_data.csv
C:\Users\alava\Downloads\data\cleaned\churn_feature_importance.csv
C:\Users\alava\Downloads\data\cleaned\churn_model_results.csv
C:\Users\alava\Downloads\data\cleaned\customer_churn_analysis.csv
C:\Users\alava\Downloads\data\cleaned\customer_cluster_summary.csv
C:\Users\alava\Downloads\data\cleaned\customer_rfm_segments.csv
C:\Users\alava\Downloads\data\cleaned\customer_segment_summary.csv
C:\Users\alava\Downloads\data\cleaned\retail_customer_analysis.csv
C:\Users\alava\Downloads\data\c

In [33]:
from pathlib import Path
import shutil

# ==============================
# SOURCE LOCATION
# ==============================

OLD_DATA = Path(r"C:\Users\alava\Downloads\data")

# ==============================
# YOUR PROJECT
# ==============================

PROJECT = Path(r"C:\Users\alava\Downloads\Online_Retail_Project")

# ==============================
# DESTINATION FOLDERS
# ==============================

CLEANED_DIR = PROJECT / "data" / "cleaned"
EDA_DIR = PROJECT / "data" / "eda"
FORECAST_DIR = PROJECT / "data" / "forecast"
DASHBOARD_DIR = PROJECT / "data" / "dashboard"

for folder in [
    CLEANED_DIR,
    EDA_DIR,
    FORECAST_DIR,
    DASHBOARD_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

# ==============================
# COPY FOLDERS
# ==============================

for source_folder, destination_folder in [
    ("cleaned", CLEANED_DIR),
    ("eda", EDA_DIR),
    ("forecast", FORECAST_DIR),
    ("dashboard", DASHBOARD_DIR)
]:
    
    source = OLD_DATA / source_folder
    
    if source.exists():
        
        for file in source.glob("*.csv"):
            
            shutil.copy2(
                file,
                destination_folder / file.name
            )
            
            print(
                f"Copied: {file.name}"
            )

print("\nAll files copied successfully.")

Copied: churn_feature_importance.csv
Copied: churn_model_results.csv
Copied: customer_churn_analysis.csv
Copied: customer_cluster_summary.csv
Copied: customer_rfm_segments.csv
Copied: customer_segment_summary.csv
Copied: retail_customer_analysis.csv
Copied: retail_demand.csv
Copied: retail_master_cleaned.csv
Copied: retail_sales.csv
Copied: country_revenue.csv
Copied: customer_revenue.csv
Copied: monthly_sales.csv
Copied: top_products.csv
Copied: 30_day_demand_forecast.csv
Copied: forecast_summary.csv
Copied: inventory_planning.csv
Copied: inventory_summary.csv
Copied: selected_product_daily_demand.csv
Copied: churn_segment.csv
Copied: churn_summary.csv
Copied: country_performance.csv
Copied: customer_segments.csv
Copied: executive_kpis.csv
Copied: forecast_dashboard.csv
Copied: high_value_at_risk.csv
Copied: inventory_dashboard.csv
Copied: monthly_sales.csv
Copied: product_performance.csv

All files copied successfully.


In [34]:
from pathlib import Path

PROJECT = Path(
    r"C:\Users\alava\Downloads\Online_Retail_Project"
)

print("CSV FILES INSIDE PROJECT")
print("=" * 60)

for file in PROJECT.rglob("*.csv"):
    print(file)

CSV FILES INSIDE PROJECT
C:\Users\alava\Downloads\Online_Retail_Project\data\cleaned\churn_feature_importance.csv
C:\Users\alava\Downloads\Online_Retail_Project\data\cleaned\churn_model_results.csv
C:\Users\alava\Downloads\Online_Retail_Project\data\cleaned\customer_churn_analysis.csv
C:\Users\alava\Downloads\Online_Retail_Project\data\cleaned\customer_cluster_summary.csv
C:\Users\alava\Downloads\Online_Retail_Project\data\cleaned\customer_rfm_segments.csv
C:\Users\alava\Downloads\Online_Retail_Project\data\cleaned\customer_segment_summary.csv
C:\Users\alava\Downloads\Online_Retail_Project\data\cleaned\retail_customer_analysis.csv
C:\Users\alava\Downloads\Online_Retail_Project\data\cleaned\retail_demand.csv
C:\Users\alava\Downloads\Online_Retail_Project\data\cleaned\retail_master_cleaned.csv
C:\Users\alava\Downloads\Online_Retail_Project\data\cleaned\retail_sales.csv
C:\Users\alava\Downloads\Online_Retail_Project\data\dashboard\churn_segment.csv
C:\Users\alava\Downloads\Online_Retail_P